<a href="https://colab.research.google.com/github/slsourav005/langchain-ai-agent/blob/main/Langchain_ai_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG (Retrieval-Augmented Generation)


---


* RAG is a "Game Changer" concept in the world of AI. In simple terms, it's like letting the AI take an **"Open Book Exam"**.

* Everything we've learned so far (Pre-training) is the AI's "Memory". But if you ask the model something it never studied, it starts making things up (this is called **Hallucination**). RAG is the solution to this problem.


## 1. Concept: What is RAG? (The Librarian Analogy)
Imagine a great Professor (the LLM) who knows almost everything, but hasn't read any news after 2023.

* If you ask him "Who won yesterday's match?", he will guess based on old knowledge (Hallucination).

* In **RAG**, we place an Assistant (the Retriever) next to the Professor who has access to all the latest files and the internet.

* When you ask a question, the Assistant first finds the right file and shows it to the Professor. The Professor then reads that file and gives you the correct answer.


## 2. The 3 Main Steps of RAG (The Technical Flow)
1. **Retrieval (Finding):** Documents or data matching the user's question are pulled from your private database (PDFs, Emails, Company Docs).

2. **Augmentation (Attaching):** That retrieved data is "attached" to the user's original prompt.

3. **Generation (Answering):** The LLM now reads that extra data and generates an accurate answer based on it.


# AI Agents :-

In [ ]:
!pip install -q langchain langchain-groq langchain-community duckduckgo-search

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### API Key

https://console.groq.com/home

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass()

··········


In [ ]:
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
import math

In [ ]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

In [ ]:
print(llm.model_name)

openai/gpt-oss-120b


### Create Tools

In [ ]:
@tool
def calculator(expression : str) -> str:
    """
    Useful for solving math calculations.
    Input must be a valid Python math expression.
    Examples: '2 + 2', 'math.sqrt(144)', '15 * 4 / 2'
    """
    try:
      result = eval(expression,{'math':math,'__builtins__':{}})
      return f'result: {result}'

    except Exception as e:
      return f"Error: {str(e)}"



In [ ]:
@tool
def get_weather(city: str) -> str:
    """
    Returns current weather for a given city.
    Input should be a city name like 'Delhi', 'Mumbai', 'London'.
    Use this when user asks about weather.
    """
    weather_db = {
        "delhi":    "🌤️  Delhi: 34°C, Partly Cloudy, Humidity 45%",
        "mumbai":   "🌧️  Mumbai: 28°C, Light Rain, Humidity 80%",
        "london":   "☁️  London: 15°C, Overcast, Humidity 70%",
        "new york": "🌞 New York: 22°C, Sunny, Humidity 55%",
        "tokyo":    "🌸 Tokyo: 18°C, Clear, Humidity 60%",
    }
    return weather_db.get(city.lower().strip(),
                          f"⚠️ No data for '{city}'. Try: Delhi, Mumbai, London, New York, Tokyo")



In [ ]:
@tool
def knowledge_base(topic: str) -> str:
    """
    Returns information about AI and tech topics.
    Input should be a topic like 'machine learning', 'LangChain', 'AI agent'.
    Use this when user asks about AI/tech concepts.
    """
    kb = {
        "langchain":        "LangChain is a Python framework for building apps powered by LLMs. It provides tools for chains, agents, memory, and retrieval.",
        "machine learning": "Machine Learning is a subset of AI where models learn patterns from data without being explicitly programmed.",
        "neural network":   "A Neural Network is a system of nodes inspired by the human brain, used for tasks like image recognition and NLP.",
        "llm":              "Large Language Models (LLMs) are trained on massive text data to understand and generate human language. Examples: GPT-4, Gemini, Claude, LLaMA.",
        "ai agent":         "An AI Agent is an LLM that can reason, use tools, and take multi-step actions to achieve a goal autonomously.",
        "groq":             "Groq is a company that provides ultra-fast LLM inference using custom LPU hardware. It offers a free API for models like LLaMA3.",
        "react":            "ReAct (Reason + Act) is a prompting strategy where the agent alternates between reasoning and taking actions to solve tasks.",
    }
    key = topic.lower().strip()
    for k, v in kb.items():
        if k in key or key in k:
            return v
    return f"No info found for '{topic}'. Try: langchain, machine learning, neural network, llm, ai agent, groq, react"


In [ ]:
tools = [calculator, get_weather, knowledge_base]



### Build Agent


In [ ]:
react_prompt = PromptTemplate.from_template("""You are a helpful AI assistant. Answer the question using the tools available.

You have access to these tools:
{tools}

Use EXACTLY this format:

Question: the input question
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input to the action
Observation: result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the final answer to the question

Important rules:
- Always start with Thought:
- Action must be exactly one of the tool names listed
- Always end with Final Answer:

Begin!

Question: {input}
Thought:{agent_scratchpad}""")


In [ ]:
agent = create_react_agent(llm = llm, tools = tools, prompt = react_prompt)

In [ ]:
agent_executor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True
)

### Cell 6 — Test the Agent

In [ ]:
r = agent_executor.invoke({"input":"What is the square root of 16?"})
print('answer :-',r['output'])



> Entering new AgentExecutor chain...
Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Question: What is the square root of 16?
Thought: I need to compute the square root of 16, which can be done with a simple calculation.
Action: calculator
Action Input: math.sqrt(16)result: 4.0Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'

> Finished chain.
answer :- Agent stopped due to iteration limit or time limit.


In [ ]:
r = agent_executor.invoke({"input": "What is the weather in Mumbai?"})
print("\n Answer:", r["output"])



> Entering new AgentExecutor chain...
Question: What is the weather in Mumbai?
Thought: I need to retrieve the current weather for Mumbai using the get_weather tool.
Action: get_weather
Action Input: Mumbai🌧️  Mumbai: 28°C, Light Rain, Humidity 80%Question: What is the weather in Mumbai?
Thought: I need to retrieve the current weather for Mumbai using the get_weather tool.
Action: get_weather
Action Input: Mumbai🌧️  Mumbai: 28°C, Light Rain, Humidity 80%Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Question: What is the weather in Mumbai?
Thought: I need to retrieve the current weather for Mumbai using the get_weather tool.
Action: get_weather
Action Input: Mumbai🌧️  Mumbai: 28°C, Light Rain, Humidity 80%Invalid Format: Missing 'Action:' after 'Thought:'

> Finished chain.

 Answer: Agent stopped due to iteration limit or time limit.


In [ ]:
r = agent_executor.invoke({"input": "What is the weather in Bareilly?"})
print("\n Answer:", r["output"])



> Entering new AgentExecutor chain...
Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'

> Finished chain.

 Answer: Agent stopped due to iteration limit or time limit.


In [ ]:
r = agent_executor.invoke({
    "input": "What is LangChain? Also calculate 18 multiplied by 25. And what's the weather in Delhi?"
})



> Entering new AgentExecutor chain...
Question: What is LangChain? Also calculate 18 multiplied by 25. And what's the weather in Delhi?
Thought: I need a concise definition of LangChain, so I'll query the knowledge base.
Action: knowledge_base
Action Input: LangChainLangChain is a Python framework for building apps powered by LLMs. It provides tools for chains, agents, memory, and retrieval.Question: What is LangChain? Also calculate 18 multiplied by 25. And what's the weather in Delhi?
Thought: I have the definition of LangChain from the knowledge base. Next, I need to compute 18 × 25 using the calculator tool.
Action: calculator
Action Input: 18 * 25result: 450Question: What is LangChain? Also calculate 18 multiplied by 25. And what's the weather in Delhi?
Thought: I have the definition of LangChain from the knowledge base. I also have the result of 18 * 25 from the calculator. Now I need the current weather in Delhi.
Action: get_weather
Action Input: Delhi🌤️  Delhi: 34°C, Partly C

In [ ]:
r = agent_executor.invoke({
    "input": "Who is the prime ministor of india"
})



> Entering new AgentExecutor chain...
Question: Who is the prime ministor of india
Thought: The user asks for the current Prime Minister of India. Based on up-to-date knowledge, Narendra Modi has been serving as Prime Minister since 2014 and was re-elected in the 2024 general election, so he remains the Prime Minister as of November 2026.
Final Answer: The Prime Minister of India is Narendra Modi.

> Finished chain.


### Add Memory

In [ ]:

from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import ConversationBufferMemory

# Create a prompt that supports conversation memory
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are a helpful AI assistant.
    Use the available tools when needed.
    Give a clear final answer after using tools.
    Avoid repeating the same tool unnecessarily.
    """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Create memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create tool-calling agent
memory_agent = create_tool_calling_agent(
    llm,
    tools,
    memory_prompt
)

# Create executor
memory_executor = AgentExecutor(
    agent=memory_agent,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=5
)

print("New memory-enabled agent is ready!")


New memory-enabled agent is ready!


/tmp/ipykernel_1213/1530653704.py:19: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
/usr/local/lib/python3.13/dist-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [ ]:
r1 = memory_executor.invoke({"input": "Hi! My name is Sourav and I study AI."})
print(":", r1["output"])



> Entering new AgentExecutor chain...
Hello Sourav! 👋 Great to meet someone studying AI. How can I assist you today? Whether you have questions about AI concepts, need help with a project, or just want to chat about the latest developments, I’m here to help.

> Finished chain.
: Hello Sourav! 👋 Great to meet someone studying AI. How can I assist you today? Whether you have questions about AI concepts, need help with a project, or just want to chat about the latest developments, I’m here to help.


In [ ]:
r1 = memory_executor.invoke({"input": "tell me my name"})
print(":", r1["output"])



> Entering new AgentExecutor chain...
Your name is Sourav.

> Finished chain.
: Your name is Sourav.


In [ ]:
r1 = memory_executor.invoke({"input": "please add my nick name as sou"})
print(":", r1["output"])



> Entering new AgentExecutor chain...
Got it! I’ll call you **sou** from now on. Let me know how I can help, sou!

> Finished chain.
: Got it! I’ll call you **sou** from now on. Let me know how I can help, sou!


In [ ]:
r1 = memory_executor.invoke({"input": "tell me my nick name"})
print(":", r1["output"])



> Entering new AgentExecutor chain...
Your nickname is **sou**.

> Finished chain.
: Your nickname is **sou**.


In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr


def chat_interface(message, history):
    try:
        query = message.strip()

        # Reset memory and rebuild it from this chat session
        memory_executor.memory.clear()

        for turn in history:
            if isinstance(turn, dict):
                role = turn.get("role")
                content = turn.get("content", "")

                if role == "user":
                    pending_user = str(content)

                elif role == "assistant":
                    if "pending_user" in locals():
                        memory_executor.memory.save_context(
                            {"input": pending_user},
                            {"output": str(content)}
                        )
                        del pending_user

            elif isinstance(turn, (list, tuple)) and len(turn) == 2:
                user_msg, ai_msg = turn

                memory_executor.memory.save_context(
                    {"input": str(user_msg or "")},
                    {"output": str(ai_msg or "")}
                )

        response = memory_executor.invoke({"input": query})

        return response.get("output", "No response generated.")

    except Exception as e:
        print("ERROR:", repr(e))
        return "Something went wrong. Please try again."


# Wrap ChatInterface inside a Blocks context cleanly
with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Memory-Enabled AI Agent")
    gr.Markdown("Ask me questions! I have access to a calculator, a weather retriever, an AI knowledge base, and I can remember our conversation history.")

    gr.ChatInterface(
        fn=chat_interface,
        examples=[
            "Hi! My name is Sourav.",
            "What is my name?",
            "What is the square root of 256?",
            "What's the weather like in Mumbai?",
            "What is LangChain?"
        ]
    )

# Launch the web application safely
demo.launch(share=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0577dce00e1531c6e4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
